In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

#version transformers==4.53.3 worked

In [2]:
!pip install transformers==4.53.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
#model_name = "microsoft/Phi-4-mini-instruct"
model_name= "meta-llama/Meta-Llama-3-8B"
#model_name= "google/gemma-2-2b"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = 'auto',
    torch_dtype = torch.float16,
    trust_remote_code = True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Necessario per il batching

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

In [ ]:
ds = load_dataset("FMiMiY/SS-GEN", split='test')

README.md:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

(…)ory_all_5085_from_gpt4titles_train.jsonl: 0.00B [00:00, ?B/s]

(…)story_all_5085_from_gpt4titles_dev.jsonl: 0.00B [00:00, ?B/s]

(…)tory_all_5085_from_gpt4titles_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4068 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/509 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/508 [00:00<?, ? examples/s]

In [ ]:
#Zero-shot
def tokenizer_zeroshot(batch):
    #template riportato dalla figura 5 del paper
    prompt = (
        "Develop a concise, clear, straightforward, positive and supportive "
        "Social Story titled \"{title}\" for children and teens with autism, "
        "200-300 words, that promotes their social understanding and boosts "
        "their participation in daily activities, fostering independence and confidence."
    )

    texts = [
        f"{prompt.format(title=t)}\nTitle: {t}\n\nSocial Story:"
        for t in batch['title']
    ]
    return {"text": texts}


In [ ]:
tokenized_data = ds.map(tokenizer_zeroshot,batched=True, remove_columns=ds.column_names)

Map:   0%|          | 0/508 [00:00<?, ? examples/s]

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import os
import torch
from tqdm.auto import tqdm

# Percorso del file su Google Drive
checkpoint_path = "/content/drive/MyDrive/SS_GEN_results_Llama3_8B.json"

# 1. Carica i risultati precedenti se esistono (RESUME)
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Rilevato checkpoint: riparto dalla storia numero {len(results)}")
else:
    results = []
    print("Nessun checkpoint trovato: inizio da capo.")

start_index = len(results)

# 2. Ciclo di generazione
for i in tqdm(range(start_index, len(tokenized_data)), desc="Generazione con Checkpoint"):

    # Prepara l'input
    inputs = tokenizer(tokenized_data[i]['text'], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

    # Aggiungi il nuovo risultato
    results.append({
        "id": i,
        "prompt": tokenized_data[i]['text'],
        "generated_story": generated_text.strip()
    })

    # 3. Salva su Drive ogni 5 iterazioni (CHECKPOINT)
    if i % 5 == 0:
        with open(checkpoint_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)

# Salvataggio finale definitivo
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"\nLavoro completato! Totale storie: {len(results)}")

Rilevato checkpoint: riparto dalla storia numero 46


Generazione con Checkpoint:   0%|          | 0/462 [00:00<?, ?it/s]

In [ ]:
import json

# Creiamo una lista di dizionari con tutti i dati
data_to_save = []
test_titles = ds['title']
for i in range(len(test_titles)):
    entry = {
        "id": i,
        "title": test_titles[i],
        "reference_story": ds['story_content'][i],
        "generated_story": results[i]  # La lista dei risultati del tuo loop
    }
    data_to_save.append(entry)

# Salvataggio su file
with open('risultati_zero_shot_Llama38B.json', 'w', encoding='utf-8') as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=4)

print("Risultati salvati con successo!")

Risultati salvati con successo!


Metriche Oggettive :
Seguono le metriche che misurano quanto la storia generata dal modello sia simile alla storia scritta dagli esperti del dataset. È possibile utilizzare la libreria evaluate di Hugging Face.
- BLEU-4 e ROUGE: confrontano la sovrapposizione di parole (n-grammi).
- BERTscore: usa gli embedding per capire se il significato è simile, anche se le parole usate sono diverse.

In [ ]:
import evaluate

rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

reference_stories = ds['story_content']
results_rouge = rouge.compute(predictions=results, references=reference_stories)
results_bert = bertscore.compute(predictions=results, references=reference_stories, lang="en")

print(f"ROUGE-L: {results_rouge['rougeL']}")
print(f"BERTScore F1 Mean: {sum(results_bert['f1'])/len(results_bert['f1'])}")